In [1]:
import numpy as np

# ==========================================
# 1. SANITY CHECKS (W y H)
# ==========================================

# Configuración del caso miniatura (m=2, n=2, k=1)
X = np.array([[5.0, 3.0], 
              [2.0, 4.0]])

# W es de dimensiones (m, k) -> (2, 1)
W = np.array([[1.0], 
              [2.0]])

# H es de dimensiones (k, n) -> (1, 2)
H = np.array([[2.0, 1.0]])

# epsilon exigido por el proyecto
epsilon = 1e-5

# Función de pérdida base: f(W,H) = 0.5 * ||X - WH||_F^2
def f_loss(W, H, X):
    residual = (W @ H) - X
    return 0.5 * np.sum(residual ** 2)

# --- SANITY CHECK PARA W ---
grad_W_analitico = ((W @ H) - X) @ H.T
grad_W_numerico = np.zeros_like(W)
m, k = W.shape

for a in range(m):
    for b in range(k):
        E = np.zeros_like(W)
        E[a, b] = 1.0
        loss_plus = f_loss(W + epsilon * E, H, X)
        loss_base = f_loss(W, H, X)
        grad_W_numerico[a, b] = (loss_plus - loss_base) / epsilon

error_relativo_W = np.abs(grad_W_analitico - grad_W_numerico) / (np.abs(grad_W_analitico) + 1e-8)
error_maximo_W = np.max(error_relativo_W)

print("--- SANITY CHECK: GRADIENTES DE W ---")
print("Gradiente Analítico W:\n", grad_W_analitico)
print("\nGradiente Numérico W:\n", grad_W_numerico)
print(f"\nError relativo máximo W: {error_maximo_W:.2e}")
if error_maximo_W < 1e-4:
    print("✓ ÉXITO: El error relativo es menor a 1e-4.\n")
else:
    print("✗ FALLO: Revisa la derivación o la implementación.\n")


# --- SANITY CHECK PARA H ---
# grad_H = W^T(WH - X)
grad_H_analitico = W.T @ ((W @ H) - X)
grad_H_numerico = np.zeros_like(H)
k_dim, n_dim = H.shape

for c in range(k_dim):
    for d in range(n_dim):
        E_H = np.zeros_like(H)
        E_H[c, d] = 1.0
        loss_plus_H = f_loss(W, H + epsilon * E_H, X)
        loss_base_H = f_loss(W, H, X)
        grad_H_numerico[c, d] = (loss_plus_H - loss_base_H) / epsilon

error_relativo_H = np.abs(grad_H_analitico - grad_H_numerico) / (np.abs(grad_H_analitico) + 1e-8)
error_maximo_H = np.max(error_relativo_H)

print("--- SANITY CHECK: GRADIENTES DE H ---")
print("Gradiente Analítico H:\n", grad_H_analitico)
print("\nGradiente Numérico H:\n", grad_H_numerico)
print(f"\nError relativo máximo H: {error_maximo_H:.2e}")
if error_maximo_H < 1e-4:
    print("✓ ÉXITO: El error relativo de H es menor a 1e-4.\n")
else:
    print("✗ FALLO: Revisa la derivación o la implementación.\n")


# ==========================================
# 2. IMPLEMENTACIÓN DE REGLAS (2.3 y 2.4)
# ==========================================

def update_block(Block, Grad, v_Block, alpha, beta, method, is_nmf=True):
    """
    Aplica la variante de descenso de gradiente (2.3) y la proyección (2.4).
    """
    if method == 'gd':
        # 2.3.1 Vanilla GD
        Block = Block - alpha * Grad
        
    elif method == 'momentum':
        # 2.3.2 Momentum GD
        v_Block = beta * v_Block + Grad
        Block = Block - alpha * v_Block
        
    elif method == 'nesterov':
        # 2.3.3 Nesterov Accelerated Gradient
        # NOTA: En el algoritmo real, calculas un Grad_look usando Block_look. 
        # Aquí asumimos que 'Grad' ya es ese gradiente adelantado.
        v_Block = beta * v_Block + Grad
        Block = Block - alpha * v_Block

    # 2.4 Non-Negativity Constraints and Projections
    if is_nmf:
        # np.maximum proyecta todo valor negativo a 0
        Block = np.maximum(Block, 0)
        
    return Block, v_Block

# --- Mini prueba de las funciones ---
print("--- PRUEBA DE OPTIMIZACIÓN (2.3 y 2.4) ---")
alpha_w = 0.01
beta = 0.9
v_W = np.zeros_like(W)

# Probando un paso de Vanilla GD con Proyección
W_new, v_W_new = update_block(W, grad_W_analitico, v_W, alpha_w, beta, method='gd', is_nmf=True)
print("W actualizado con Vanilla GD y Proyectado:\n", W_new)

--- SANITY CHECK: GRADIENTES DE W ---
Gradiente Analítico W:
 [[-8.]
 [ 2.]]

Gradiente Numérico W:
 [[-7.999975]
 [ 2.000025]]

Error relativo máximo W: 1.25e-05
✓ ÉXITO: El error relativo es menor a 1e-4.

--- SANITY CHECK: GRADIENTES DE H ---
Gradiente Analítico H:
 [[ 1. -6.]]

Gradiente Numérico H:
 [[ 1.000025 -5.999975]]

Error relativo máximo H: 2.50e-05
✓ ÉXITO: El error relativo de H es menor a 1e-4.

--- PRUEBA DE OPTIMIZACIÓN (2.3 y 2.4) ---
W actualizado con Vanilla GD y Proyectado:
 [[1.08]
 [1.98]]
